# U0.L5 — Solvers by hand

**Session U0.L5, part 1 of 2.** Part 2 is `jax-primer.ipynb`.

This notebook builds `common/odesolve.py`: Euler, Heun, RK4 and an adaptive Heun,
each of them reporting what it cost. Then it measures them.

The module is not a demonstration. **U3.L1 imports it to draw the first samples
of this course's first Flow Matching model.** The pictures of your first
generative model will be drawn by the solver you write here.

### Before you start

Everything runs on CPU and needs only `torch`, `numpy` and `matplotlib`. If you
are reading this as part of the PESC self-study packet, read the U0.T5 notes
first — this notebook implements them and does not re-derive them.

### The frozen contract

```
odeint(f, x0, t0, t1, method="rk4", n_steps=None,
       rtol=None, atol=None, return_traj=False)
    -> x1          # a Tensor carrying .nfe
    -> (ts, xs)    # if return_traj=True
```

1. `f(t, x)` — **time first**. This is the convention of `torchdiffeq` and
   diffrax, so a right-hand side you write here is one they accept unchanged.
2. `x0` has shape `(B, d)`. The state is batched by construction.
3. **Every call reports its NFE** — the number of evaluations of `f`. NFE is
   this course's cost currency; a solver that hides it cannot be compared with
   another solver.

In [ ]:
import sys
from pathlib import Path

# The repository root, so `import common.odesolve` works from labs/.
ROOT = Path.cwd().parent if Path.cwd().name == "labs" else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import torch

torch.manual_seed(0)
np.random.seed(0)
torch.set_default_dtype(torch.float64)   # solver accuracy study: use float64

print("torch", torch.__version__)

## The test problem

The harmonic oscillator of U0.T5. It is chosen for one reason: **its exact
solution is known**, so the error of a numerical solution can be computed
without a reference solver and without a reference error.

$$
\frac{d}{dt}\begin{pmatrix} x \\ v \end{pmatrix}
= \begin{pmatrix} v \\ -x \end{pmatrix},
\qquad
\begin{pmatrix} x(0) \\ v(0)\end{pmatrix} = \begin{pmatrix} 1 \\ 0 \end{pmatrix}.
$$

The exact trajectory is the unit circle, traced forever, and the energy
$E = \tfrac12(x^2 + v^2)$ is exactly conserved. So the global error after one
period is simply the distance back to the starting point.

In [ ]:
def harmonic(t, x):
    """dx/dt = v, dv/dt = -x. TIME FIRST; state is (B, 2)."""
    return torch.stack([x[:, 1], -x[:, 0]], dim=1)


X0 = torch.tensor([[1.0, 0.0]])
T_END = 2 * torch.pi


def global_error(x1):
    """Distance back to the start after one full period."""
    return float((x1 - X0).norm())

## TODO 1 — Explicit Euler

$$x_{n+1} = x_n + h\, f(t_n, x_n)$$

One evaluation of $f$ per step. Return the new state **and** the number of
evaluations you spent, because the caller adds them up.

In [ ]:
def step_euler(f, t, x, h):
    """One Euler step. Returns (x_next, n_evaluations)."""
    # TODO 1: one evaluation, one line of arithmetic.
    raise NotImplementedError

## TODO 2 — Heun

Euler's error comes from using the velocity at the *start* of the step for the
whole step. A better estimate averages the velocity over the step — but the
velocity at the end needs the endpoint, which is what you are solving for.

So guess it, then correct:

$$
k_1 = f(t_n, x_n), \quad
k_2 = f(t_n + h,\, x_n + h k_1), \quad
x_{n+1} = x_n + \tfrac{h}{2}(k_1 + k_2).
$$

In [ ]:
def step_heun(f, t, x, h):
    """One Heun step. Returns (x_next, n_evaluations)."""
    # TODO 2: predict with Euler, then average the two velocities.
    raise NotImplementedError

## TODO 3 — RK4

Transcribe the tableau from the U0.T5 notes. This one is mechanical on purpose:
the interesting part of RK4 is not its arithmetic but its slope, and you measure
that further down.

$$
\begin{aligned}
k_1 &= f(t_n, x_n), &
k_2 &= f(t_n + \tfrac{h}{2},\, x_n + \tfrac{h}{2}k_1), \\
k_3 &= f(t_n + \tfrac{h}{2},\, x_n + \tfrac{h}{2}k_2), &
k_4 &= f(t_n + h,\, x_n + h k_3),
\end{aligned}
$$

$$x_{n+1} = x_n + \tfrac{h}{6}(k_1 + 2k_2 + 2k_3 + k_4).$$

The weights $\tfrac16, \tfrac26, \tfrac26, \tfrac16$ are Simpson's rule, and
that is not a coincidence.

In [ ]:
def step_rk4(f, t, x, h):
    """One classical RK4 step. Returns (x_next, n_evaluations)."""
    # TODO 3: four stages, then the Simpson weights.
    raise NotImplementedError

### Checkpoint 1 — the three steppers agree at small $h$

A single step of each method, from the same point, with a tiny step size. They
must all agree with the exact solution to their own order. This catches a typo
in a tableau before it becomes a mysterious slope later.

In [ ]:
h = 1e-3
t0 = torch.tensor(0.0)
exact = torch.tensor([[np.cos(h), -np.sin(h)]])

for name, step, expect in (("euler", step_euler, 1),
                           ("heun", step_heun, 2),
                           ("rk4", step_rk4, 4)):
    x1, used = step(harmonic, t0, X0, torch.tensor(h))
    err = float((x1 - exact).norm())
    assert used == {"euler": 1, "heun": 2, "rk4": 4}[name], "wrong NFE per step"
    # Local error should be about h^(p+1). Check the exponent, loosely.
    order = np.log10(err) / np.log10(h) if err > 0 else np.inf
    print(f"{name:>6s}: NFE/step {used}, local error {err:.2e}, "
          f"implied exponent {order:.1f} (expect ~{expect + 1})")
    assert order > expect + 0.5, f"{name} looks lower-order than it should"

print("\nCheckpoint 1 passed.")

## TODO 4 — Adaptive Heun(2)/Euler(1)

The only TODO with a decision in it.

Both estimates come out of the **same two evaluations**, because $k_1$ alone is
already the Euler step:

$$
\hat{x}_{n+1} = x_n + h k_1 \ \ (\text{order }1),
\qquad
x_{n+1} = x_n + \tfrac{h}{2}(k_1+k_2) \ \ (\text{order }2).
$$

Their difference estimates the local error. Judge it against the scale

$$\mathrm{sc}_i = \texttt{atol} + \texttt{rtol}\cdot\max(|x_{n,i}|, |x_{n+1,i}|),$$

and accept when the root-mean-square of $\varepsilon_i / \mathrm{sc}_i$ is at
most 1.

**Two things that are easy to get wrong.**

1. **Propagate the higher-order value.** You computed it; keep it. The
   alternative throws away the better of the two estimates you paid for.
2. **Update the step size after a rejection too.** A rejected step that retries
   at the same size rejects again, forever. The controller's rule is
   unconditional:

$$h_{\text{new}} = h \cdot \min\!\big(5,\ \max(0.2,\ 0.9\,\mathrm{err}^{-1/2})\big).$$

In [ ]:
def adaptive_heun(f, x0, t0, t1, rtol, atol, max_steps=1_000_000):
    """Embedded Heun(2)/Euler(1). Returns (x1, nfe, accepted, rejected)."""
    x, t = x0, float(t0)
    h = (t1 - t0) / 100.0
    nfe = accepted = rejected = 0

    while t < t1:
        if t + h > t1:
            h = t1 - t
        t_t, h_t = torch.tensor(t), torch.tensor(h)

        # TODO 4a: two evaluations; build the order-1 and order-2 estimates.
        raise NotImplementedError

        # TODO 4b: the error scale and the normalised error.
        # TODO 4c: accept or reject, then update h UNCONDITIONALLY.

    return x, nfe, accepted, rejected

### Checkpoint 2 — where adaptivity pays, measured

The claim to check is **not** "adaptive is better". Adaptivity is a way of
spending evaluations where the field is hard and saving them where it is easy.
On a problem that is equally hard everywhere there is nothing to save, and the
honest expectation is a saving of about $1\times$. So the checkpoint measures
two problems, at the **same achieved error**, against a fixed-step Heun:

1. the harmonic oscillator — uniform difficulty;
2. the same oscillator with a **burst**: the angular speed is $1$ everywhere
   except in a window of width $\approx 0.1$ around $t = \pi$, where it rises to
   $41$. Outside the window it *is* the harmonic oscillator; inside, the field
   is forty times harder to resolve.

$$
\frac{d}{dt}\begin{pmatrix} x \\ v \end{pmatrix}
= \omega(t) \begin{pmatrix} v \\ -x \end{pmatrix},
\qquad
\omega(t) = 1 + 40\, e^{-\left((t - \pi)/0.1\right)^2}.
$$

The burst problem keeps the property that made the oscillator the test problem:
its exact solution is known. The trajectory is a rotation by the *integrated*
angular speed, $\theta(t) = \int_0^t \omega$, and that integral is an error
function, so the global error is computed against the truth and not against a
reference solver.

In [ ]:
T_STAR, W, A = float(torch.pi), 0.1, 40.0


def omega(t):
    return 1.0 + A * torch.exp(-((t - T_STAR) / W) ** 2)


def burst(t, x):
    """The oscillator with a burst of fast rotation. TIME FIRST; (B, 2)."""
    return omega(t) * torch.stack([x[:, 1], -x[:, 0]], dim=1)


def burst_exact(t):
    """Rotation by theta(t) = t + integral of the Gaussian bump (an erf)."""
    t = torch.as_tensor(float(t))
    theta = t + A * W * torch.pi ** 0.5 / 2 * (
        torch.erf((t - T_STAR) / W) + torch.erf(torch.tensor(T_STAR / W)))
    return torch.stack([torch.cos(theta), -torch.sin(theta)]).reshape(1, 2)


def fixed_heun_at_matched_error(f, target, exact):
    """Smallest fixed-step Heun (geometric search) with error <= target."""
    n = 16
    while True:
        x, nfe = X0, 0
        h = torch.tensor(float(T_END) / n)
        for i in range(n):
            x, used = step_heun(f, torch.tensor(float(T_END) * i / n), x, h)
            nfe += used
        err = float((x - exact).norm())
        if err <= target:
            return n, nfe, err
        n = int(n * 1.25) + 1


ratios = {}
for name, f, exact in [("harmonic", harmonic, X0),
                       ("burst", burst, burst_exact(T_END))]:
    x1, nfe_ad, acc, rej = adaptive_heun(f, X0, 0.0, float(T_END),
                                         rtol=1e-4, atol=1e-6)
    err_ad = float((x1 - exact).norm())
    n, nfe_fx, err_fx = fixed_heun_at_matched_error(f, err_ad, exact)
    ratios[name] = nfe_fx / nfe_ad
    print(f"{name:9s} adaptive: NFE {nfe_ad:6d}  error {err_ad:.2e}  "
          f"({acc} accepted, {rej} rejected)")
    print(f"{'':9s} fixed   : NFE {nfe_fx:6d}  error {err_fx:.2e}  "
          f"({n} steps)   ->  fixed/adaptive = {ratios[name]:.2f}x\n")

# Both directions are the claim. A large saving on the uniform problem would
# mean the fixed-step search is wrong; no saving on the burst problem would
# mean the controller is not adapting.
assert ratios["harmonic"] < 1.5, "a uniform problem has nothing to save"
assert ratios["burst"] > 3.0, "adaptivity did not pay off — check the controller"
print("Checkpoint 2 passed: the saving is a property of the problem.")

Read the two ratios together. On the harmonic oscillator the fixed step was
already the right step everywhere, and the adaptive method spent about the same.
On the burst problem the fixed step had to be small enough for the window and
was then wasted on the other $95\%$ of the interval; the adaptive method took
small steps only where $\omega$ was large. **Adaptivity pays where the difficulty
is nonuniform**, and that is a measurement, not a theorem. It is also the
situation of a learned velocity field: nobody promises that its difficulty is
spread evenly along the trajectory, and U2.L2 measures what it actually does.

### Checkpoint 3 — your own spiral

Now reproduce `fig-euler-spiral`, the icon of U0.T5, with your own solvers.

The check is **not** that the plot looks right. Euler on this problem multiplies
the energy by exactly $1 + h^2$ per step — that is a theorem, not an artifact —
so the assert is on the growth factor. A plausible-looking spiral with the wrong
growth factor is a bug, and the eye does not catch it.

In [ ]:
def integrate(step, n, T=3 * float(T_END)):
    h = torch.tensor(T / n)
    xs, x = [X0], X0
    for i in range(n):
        x, _ = step(harmonic, torch.tensor(T * i / n), x, h)
        xs.append(x)
    return torch.cat(xs, dim=0)


h_val = float(T_END) / 40
n_steps = int(3 * float(T_END) / h_val)
traj_euler = integrate(step_euler, n_steps)
traj_rk4 = integrate(step_rk4, n_steps)

energy = 0.5 * (traj_euler[:, 0] ** 2 + traj_euler[:, 1] ** 2)
ratios = (energy[1:] / energy[:-1]).numpy()
predicted = 1 + h_val ** 2
print(f"measured energy growth per step: {ratios.mean():.10f}")
print(f"predicted (1 + h^2)            : {predicted:.10f}")
assert np.allclose(ratios, predicted, rtol=1e-10), \
    "Euler's energy growth is not (1 + h^2) — there is a bug in step_euler"

fig, ax = plt.subplots(figsize=(4.6, 4.6))
theta = np.linspace(0, 2 * np.pi, 400)
ax.plot(np.cos(theta), np.sin(theta), "--", color="#9aa2aa", lw=1,
        label="exact orbit")
ax.plot(traj_rk4[:, 0], traj_rk4[:, 1], color="#003366", lw=1.4, label="RK4")
ax.plot(traj_euler[:, 0], traj_euler[:, 1], color="#d97a1f", lw=1.2,
        label="Euler")
ax.set_aspect("equal"); ax.legend(frameon=False); ax.set_xlabel("x")
ax.set_ylabel("v"); ax.set_title("three periods, h = 2*pi/40")
plt.show()

print("\nCheckpoint 3 passed.")

## The module

Everything above now lives in `common/odesolve.py`, behind the frozen `odeint`
contract. Import it and check that the package agrees with what you just wrote.

Note what `odeint` returns: a Tensor that **carries its own cost**. Same trick as
`FIDValue` in `common/eval.py` from U0.L4 — the value is the thing the contract
promises, and it knows how it was produced. One honest limitation: the
attributes do not survive arithmetic, so read `.nfe` from the returned value.

In [ ]:
from common.odesolve import odeint

x1 = odeint(harmonic, X0, 0.0, float(T_END), method="rk4", n_steps=1000)
print(x1.report())
print("type:", type(x1).__mro__[1].__name__, "| shape:", tuple(x1.shape))

mine, _ = step_rk4(harmonic, torch.tensor(0.0), X0, torch.tensor(1e-3))
theirs = odeint(harmonic, X0, 0.0, 1e-3, method="rk4", n_steps=1)
assert torch.allclose(mine, theirs, atol=1e-14), \
    "your RK4 and the module's disagree"
print("\nYour steppers and the module agree.")

## TODO 5 — The convergence study

Exercise 2 of U0.T5 asked you to predict the three slopes of a log-log plot of
global error against step size, **before** making the plot. The answer was 1, 2
and 4. Now make the plot.

This is not a ritual. **Measuring the order is the standard way to find a bug in
a Runge–Kutta method**, because a transcription error in a tableau usually
leaves a method that still converges — at the wrong rate. A hand-written RK4
that scores slope 3 has a typo, and the slope is what finds it.

In [ ]:
methods = ["euler", "heun", "rk4"]
n_list = [25, 50, 100, 200, 400, 800, 1600]
results = {}

# TODO 5: for each method and each n, integrate to T_END and record
# (h, error, nfe). Then fit a straight line to log(error) against log(h)
# and assert the slope is the order.
raise NotImplementedError

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
colors = {"euler": "#d97a1f", "heun": "#6b7280", "rk4": "#003366"}

for method, (hs, errs, nfes) in results.items():
    axes[0].loglog(hs, errs, "o-", color=colors[method], label=method)
    axes[1].loglog(nfes, errs, "o-", color=colors[method], label=method)

axes[0].set_xlabel("step size h"); axes[0].set_ylabel("global error")
axes[0].set_title("(a) error against h — the slope is the order")
axes[1].set_xlabel("NFE"); axes[1].set_ylabel("global error")
axes[1].set_title("(b) error against NFE — the honest cost axis")
for ax in axes:
    ax.legend(frameon=False)
    ax.grid(color="#DDDDDD", lw=0.6)
plt.tight_layout(); plt.show()

### The accounting table, and one decision

Panel (b) is the honest comparison, because a step of RK4 costs four times a
step of Euler. Here is the same point stated as sharply as it can be: **the same
budget of 4000 evaluations, spent three ways.**

Run the cell, then answer the question below it in writing. The question is the
point of this block — the NFE-against-accuracy trade is now a decision you make,
not a fact you are told.

In [ ]:
print(f"{'method':>8s} {'steps':>7s} {'NFE':>7s} {'error':>12s}")
for method, n in (("euler", 4000), ("heun", 2000), ("rk4", 1000)):
    x1 = odeint(harmonic, X0, 0.0, float(T_END), method=method, n_steps=n)
    assert int(x1.nfe) == 4000
    print(f"{method:>8s} {n:>7d} {int(x1.nfe):>7d} {global_error(x1):>12.2e}")

**Question.** At an error budget of $10^{-4}$, which method do you choose, and
how many evaluations does it cost? Read it off panel (b).

**Second question, and it matters later.** The table above assumes you want high
accuracy. At a *loose* tolerance the three lines in panel (b) sit much closer
together, and at the extreme — one or two evaluations in total — a high-order
method has no advantage at all, because it has not taken enough steps for its
order to express itself.

Sampling a generative model at 4 NFE lives in exactly that regime. That is why
the few-step sampling literature is not simply "use RK4", and U3.T4 returns to
it.

*Write your answers here.*

## TODO 6 — The tolerance sweep

The adaptive method does not take a step size. It takes a pair of tolerances and
decides. So the study to run on it is a different one: sweep the request and see
what you get.

In [ ]:
rtols = np.logspace(-1, -8, 8)

# TODO 6: for each rtol (with atol = rtol/100), run the adaptive method to
# T_END and record NFE, the achieved error, and the accepted/rejected counts.
# Then plot NFE against the achieved error on log-log axes.
raise NotImplementedError

### The two failure smells, on your own plot

U0.T5 named two ways of getting the tolerances wrong. Both are on the plot you
just made.

**Too loose** is the left end: about 32 evaluations, and an error of $0.2$ on a
trajectory whose whole radius is 1. The solver returned quickly, and it returned
a smooth, plausible, entirely wrong orbit. **Nothing complained.** This is the
dangerous failure, and the test for it is mechanical: halve both tolerances and
re-run. If the answer moves by more than you can accept, the original answer was
not converged.

**Too tight** is the right end: about 97 000 evaluations for eight correct
digits, a cost three thousand times the loose run. On a toy problem that is a few
seconds; on a learned velocity field it is the difference between sampling in a
minute and sampling over lunch. This is the cheap failure — you notice it
immediately, and it costs only time.

Run the halving test yourself:

In [ ]:
for rtol in (1e-2, 5e-3):
    x1 = odeint(harmonic, X0, 0.0, float(T_END), method="adaptive_heun",
                rtol=rtol, atol=rtol / 100)
    print(f"rtol {rtol:.0e}: x1 = {x1[0].tolist()}  (NFE {x1.nfe})")

print("\nDid the answer move by more than you can accept?")

## The freeze

`common/odesolve.py` is **frozen** at the end of this session, and tagged
`u0l5-freeze`. U3.L1 imports `odeint` under the contract at the top of this
notebook, so a change to that signature after today is a breaking change and
needs a dated note.

### What this session hands forward

| From here | Reappears in |
|---|---|
| `odeint`, NFE as a returned value | **U3.L1** — sampling a Flow Matching model |
| the convergence study | U2.L2 — solver choice under a real field |
| tolerance discipline | U2.L1 — where the solver is somebody else's |

**Next:** `jax-primer.ipynb`, and then the same problem in diffrax.